# SMARTGRID-X -- Phase 0: Exploring the raw data

**Project:** occupancy-aware energy-waste and anomaly analysis of the I-BLEND
campus dataset (IIIT Delhi, 7 buildings).

**What this notebook does.** Before cleaning anything or fitting any model, we
look at what we actually have. This notebook opens every single file in the
`Dataset` folder, prints the first and last few rows, measures how much of the
time range each meter really covers, and counts the data problems we will have
to deal with in Phase 1.

**Why do this first.** The energy folder is about 1.6 GB of 1-minute readings.
If we start writing analysis code before we know where the gaps and the bad
values are, every number we produce later is untrustworthy. Half of a data
science project is finding out what is wrong with the data.

**The rule we work under:** `Dataset/` is read-only source data. Nothing in
this project ever writes to it, and it is never committed to GitHub.

**What this notebook produces:**

* `results/phase0_energy_profile.csv` -- one row per energy meter
* `results/phase0_occupancy_profile.csv` -- one row per building
* `figures/fig_00_coverage_timeline.png` -- when each meter was recording
* Section 4 (Dataset & data quality) of `docs/PROJECT_REPORT.md`

## Step 0: the Python environment

Part of the syllabus is being able to say what our tools are and why. We print
the versions from the running interpreter rather than typing them by hand, so
this record can never drift out of date.

### Opening this notebook in Colab

Colab hands you this `.ipynb` and nothing else -- no `src/`, no data. The cell
below clones the repository, installs the portable dependency set and fetches
the raw dataset (1.6 GB from figshare).

It does nothing at all outside Colab, so running this notebook locally, or
through `tools/build_and_run.py`, is unaffected.

In [1]:
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO = Path("/content/Smartgrid")
    if not REPO.is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/rajshinde0/Smartgrid.git", str(REPO)],
            check=True,
        )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "-r", str(REPO / "requirements-portable.txt")],
            check=True,
        )
        # Plain, not --all: the extra I-BLEND file is a 2018 weather record
        # with no overlap at all with the 2014-2017 analysis window. The report
        # discusses it; no phase loads it.
        subprocess.run(
            [sys.executable, str(REPO / "tools" / "get_data.py")],
            check=True,
        )
    print("Colab: repository and data ready")

In [2]:
import sys
from pathlib import Path

# Find the project root. Locally, nbconvert runs this with the working directory
# set to notebooks/, so the root is one level up. In Colab everything runs from
# /content with the clone one level down. Looking here, above and below covers
# both, so neither case has to know about the other.
_here = Path.cwd()
_search = [_here, *_here.parents, *(p for p in _here.iterdir() if p.is_dir())]
ROOT = next((p for p in _search if (p / "src" / "config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError(f"SMARTGRID-X project root not found from {_here}")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
import sklearn
import scipy
import pyarrow

from IPython.display import display, Markdown

from src import config as C
from src import explore, viz

viz.setup_style()
C.ensure_dirs()

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

why = {
    "python": (sys.version.split()[0], "the language everything is written in"),
    "pandas": (pd.__version__, "tables: reading CSVs, resampling, grouping, merging"),
    "numpy": (np.__version__, "arrays and the matrix maths behind PCA"),
    "pyarrow": (pyarrow.__version__, "reads/writes the parquet cache we build in Phase 1"),
    "matplotlib": (matplotlib.__version__, "every chart in the report"),
    "scipy": (scipy.__version__, "distribution fitting and hypothesis tests"),
    "scikit-learn": (sklearn.__version__, "regression, PCA, k-means, train/test splits"),
}
env = pd.DataFrame(
    [{"tool": k, "version": v[0], "what we use it for": v[1]} for k, v in why.items()]
)
display(env)
print("project root:", C.PROJECT_ROOT)

,tool,version,what we use it for
0,python,3.14.3,the language everything is written in
1,pandas,3.0.3,"tables: reading CSVs, resampling, grouping, me..."
2,numpy,2.4.3,arrays and the matrix maths behind PCA
3,pyarrow,24.0.0,reads/writes the parquet cache we build in Pha...
4,matplotlib,3.10.8,every chart in the report
5,scipy,1.17.1,distribution fitting and hypothesis tests
6,scikit-learn,1.8.0,"regression, PCA, k-means, train/test splits"


project root: C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP


## Step 1: what is actually in the folder

We list every file with its size. This is the first reality check: the numbers
below are why we cannot simply load everything into memory at once.

In [3]:
print("Dataset/ top level")
display(explore.file_inventory(C.DATASET_DIR))

print("\nDataset/energy_dataset/ -- 1-minute electrical readings")
energy_files = explore.file_inventory(C.ENERGY_DIR)
display(energy_files)
print(f"total energy folder: {energy_files['size_mb'].sum():,.0f} MB")

print("\nDataset/IIITD_occupancy_dataset/ -- 10-minute occupancy counts")
occ_files = explore.file_inventory(C.OCCUPANCY_DIR)
display(occ_files)
print(f"total occupancy folder: {occ_files['size_mb'].sum():,.0f} MB")

Dataset/ top level


,file,size_mb
0,IIITD_and_airport_data.csv,0.22
1,calender_year_2014_.csv,0.01
2,.DS_Store,0.01
3,calender_year_2017_.csv,0.01
4,calender_year_2015_.csv,0.01
5,calender_year_2016_.csv,0.01
6,a_Rashid372C_assay.txt,0.00
7,calender_year_2013_.csv,0.00
8,i_Investigation.txt,0.00
9,ReadMe.txt,0.00



Dataset/energy_dataset/ -- 1-minute electrical readings


,file,size_mb
0,all_buildings_power.csv,187.81
1,acad_build_mains.csv,125.16
2,girls_hostel_ups.csv,123.82
3,mess_build_mains.csv,111.26
4,facilities_build_mains.csv,110.70
5,transformer_2.csv,95.68
6,transformer_3.csv,95.17
7,boys_hostel_ups.csv,93.18
8,library_build_mains.csv,92.70
9,girls_hostel_mains.csv,89.95


total energy folder: 1,574 MB

Dataset/IIITD_occupancy_dataset/ -- 10-minute occupancy counts


,file,size_mb
0,BH.csv,3.22
1,GH.csv,3.16
2,ACB.csv,3.12
3,DB.csv,3.07
4,LB.csv,2.94
5,SRB.csv,2.59
6,LCB.csv,2.54
7,.DS_Store,0.01
8,Readme.txt,0.00


total occupancy folder: 21 MB


**What we found.** The energy folder is about 1.5 GB spread over 17 files; the
occupancy folder is only about 21 MB. Two energy files are much bigger than the
rest because they hold every building side by side:
`all_buildings_power.csv` (power only, all 9 meters) and
`data_present_status_buildings.csv` (a 1/0 "is this reading present" flag per
minute per building). Those two will be useful later as an independent
cross-check and as the source of our missing-data heatmap.

## Step 2: what the dataset authors tell us

Both folders ship a `Readme.txt`, and the dataset also comes with ISA-Tab
metadata files describing how it was collected. Reading these first saves a lot
of guessing. The two facts that matter most:

1. Timestamps are UNIX seconds and **must** be read as `Asia/Kolkata` (+05:30).
2. `power` is in watts.

In [4]:
print("=" * 78)
print("energy_dataset/Readme.txt")
print("=" * 78)
print((C.ENERGY_DIR / "Readme.txt").read_text(encoding="utf-8", errors="replace"))

energy_dataset/Readme.txt
This folder contains 16 CSV files. Data description of each file is:

---------------------------------------------------------------------------------------------------------------------------
1. acad_build.csv:  This file contains the energy consumption data of Academic building.

2. boys_hostel_ups.csv:  This file contains the energy consumption data of backup supply to boys dormitory. Since, dormitories have both mains and UPS supply so separate meters are used to measure each type.

3. boys_hostel_mains.csv:  This file contains the energy consumption data of mains supply to boys dormitory.

4. girls_hostel_ups.csv:  This file contains the energy consumption data of backup supply to girls dormitory. 

5. girls_hostel_mains.csv:  This file contains the energy consumption data of mains supply to girls dormitory.

6. library_build_mains.csv:  This file contains the energy consumption data of Library building on campus. 

7. mess_build_mains.csv:  This file co

In [5]:
print("=" * 78)
print("IIITD_occupancy_dataset/Readme.txt")
print("=" * 78)
print((C.OCCUPANCY_DIR / "Readme.txt").read_text(encoding="utf-8", errors="replace"))

IIITD_occupancy_dataset/Readme.txt


---------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------

This folder contains 7 CSV files corresponding to seven buildings. Names are as

1. ACB : Academic Building
2. BH : Boy's Dormitory Buidling
3. DB : Dining Building
4. GH : Girl's Dormitory Building
5. LB : Library Building
6. LCB : Lecture Building
7. SRB : Facilties Building

---------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------
1. Please refer to paper for further building details. The time

### How the data was measured (provenance)

The ISA-Tab metadata that ships with the dataset records the instruments used.
This is worth quoting in the report because it tells us what kind of errors to
expect: a panel meter can lose communications (giving gaps), and a WiFi access
point counts *devices associated to it*, not people.

In [6]:
assay = pd.read_csv(C.DATASET_DIR / "a_Rashid372C_assay.txt", sep="\t")
provenance = (
    assay[["Sample Name", "Protocol REF", "Parameter Value[instrument]",
           "Parameter Value[manufacturer]"]]
    .drop_duplicates(subset=["Protocol REF"])
    .reset_index(drop=True)
)
display(provenance)

study = (C.DATASET_DIR / "i_Investigation.txt").read_text(encoding="utf-8", errors="replace")
for line in study.splitlines():
    if line.startswith(("Study Title", "Study Identifier", "Study Public Release Date",
                        "Comment[Data Record URI]")):
        print(line)

,Sample Name,Protocol REF,Parameter Value[instrument],Parameter Value[manufacturer]
0,ACB,Data collection from building level meters,EM6400 panel meter,Schneider
1,ACB,Occupancy data,Access Point,Cisco


Study Identifier	10.1038/sdata.2019.15
Study Title	I-BLEND, a campus-scale commercial and residential buildings electrical energy dataset
Study Public Release Date	19/02/2019
Comment[Data Record URI]	https://doi.org/10.6084/m9.figshare.c.3893581


## Step 3: the mapping that makes this project possible

The energy meters and the occupancy files use different names. The two
`Readme.txt` files give us the mapping, and it is the single most important
piece of setup in the project -- without it we cannot put power and people on
the same row.

Note that the two dormitories have **two** meters each: a `mains` supply and a
`ups` (backup) supply. We keep them separate and also add them together,
because "which supply keeps running when the rooms empty out" is one of our
Phase 5 questions.

In [7]:
mapping = pd.DataFrame(
    [
        {
            "building": name,
            "label": spec["label"],
            "occupancy file": f"{spec['occ_code']}.csv",
            "energy meters": ", ".join(C.METERS[m] for m in spec["meters"]),
            "kind": spec["kind"],
        }
        for name, spec in C.BUILDINGS.items()
    ]
)
display(mapping)

,building,label,occupancy file,energy meters,kind
0,Academic,Academic Building,ACB.csv,acad_build_mains.csv,commercial
1,Boys_Hostel,Boys Dormitory,BH.csv,"boys_hostel_mains.csv, boys_hostel_ups.csv",residential
2,Girls_Hostel,Girls Dormitory,GH.csv,"girls_hostel_mains.csv, girls_hostel_ups.csv",residential
3,Mess,Dining Building (Mess),DB.csv,mess_build_mains.csv,commercial
4,Library,Library Building,LB.csv,library_build_mains.csv,commercial
5,Lecture,Lecture Building,LCB.csv,lecture_build_mains.csv,commercial
6,Facilities,Facilities Building,SRB.csv,facilities_build_mains.csv,commercial


## Step 4: first and last rows of every energy meter

Here we use a `for` loop over the meter registry defined in `src/config.py`,
with an `if` test to report which optional columns each file has. Writing it as
a loop over a dictionary -- rather than nine copy-pasted blocks -- is the point:
add a meter to the registry and everything downstream picks it up.

`head_tail()` reads the file in chunks and keeps only the last chunk, so we can
see the final rows of a 125 MB file without ever holding it all in memory.

In [8]:
for meter_key, filename in C.METERS.items():
    path = C.meter_path(meter_key)
    head, tail = explore.head_tail(path, n=3)

    building = C.building_of_meter(meter_key)
    print("=" * 78)
    print(f"{meter_key}  ({filename})   building: {building}")
    print("=" * 78)
    print("columns:", list(head.columns))

    # Not every file carries every column -- transformers have no frequency.
    if "frequency" in head.columns:
        print("  has frequency column: yes")
    else:
        print("  has frequency column: no")

    print("\nfirst 3 rows:")
    print(head.to_string(index=False))
    print("\nlast 3 rows:")
    print(tail.to_string(index=False))
    print()

acad_mains  (acad_build_mains.csv)   building: Academic
columns: ['timestamp', 'power', 'current', 'voltage', 'frequency', 'power_factor']
  has frequency column: yes

first 3 rows:
 timestamp       power  current   voltage  frequency  power_factor
1376073000 28368.60645      NaN 240.99646   50.32322       0.93397
1376073060 28411.98828      NaN 241.19509   50.35931       0.93536
1376073120 28752.53809      NaN 241.38574   50.39784       0.93675

last 3 rows:
   timestamp       power  current   voltage  frequency  power_factor
1514744820.0 17106.15381 26.50895 239.38129   50.00833       0.90187
1514744880.0 18226.02930 28.06522 239.18339   50.01891       0.90786
1514744940.0 18173.18750 28.01112 239.27480   50.02542       0.90726



boys_mains  (boys_hostel_mains.csv)   building: Boys_Hostel
columns: ['timestamp', 'power', 'current', 'voltage', 'frequency', 'power_factor']
  has frequency column: yes

first 3 rows:
 timestamp       power  current   voltage  frequency  power_factor
1376073000 28298.05859      NaN 243.54842   50.33528       0.96101
1376073060 28327.11426      NaN 243.72989   50.36023       0.96289
1376073120 28343.01562      NaN 243.94641   50.41634       0.96056

last 3 rows:
   timestamp       power  current   voltage  frequency  power_factor
1514744820.0 29398.30371 43.24609 241.43561   50.01349       0.96136
1514744880.0 31212.18750 45.66242 241.24714   50.02248       0.95721
1514744940.0 31190.19531 45.62700 241.29840   50.02537       0.95714



boys_ups  (boys_hostel_ups.csv)   building: Boys_Hostel
columns: ['timestamp', 'power', 'current', 'voltage', 'frequency', 'power_factor']
  has frequency column: yes

first 3 rows:
 timestamp       power  current   voltage  frequency  power_factor
1376073000 11521.14258      NaN 243.76941   50.33707      -0.93842
1376073060 11532.83105      NaN 243.95998   50.36029      -0.94006
1376073120 11538.75635      NaN 244.17917   50.41882      -0.93904

last 3 rows:
   timestamp       power  current   voltage  frequency  power_factor
1514744820.0 10431.51221 17.32088 242.16668   50.01423       0.90129
1514744880.0 10478.02148 17.40474 242.01179   50.02126       0.96729
1514744940.0 10499.16895 17.49004 242.08063   50.02618       0.96698



girls_mains  (girls_hostel_mains.csv)   building: Girls_Hostel
columns: ['timestamp', 'power', 'current', 'voltage', 'frequency', 'power_factor']
  has frequency column: yes

first 3 rows:
 timestamp      power  current   voltage  frequency  power_factor
1376073000 9817.66846      NaN 243.94061   50.33109      -0.99601
1376073060 9804.43408      NaN 244.11749   50.35330      -0.99611
1376073120 9834.51758      NaN 244.34159   50.40518      -0.99640

last 3 rows:
   timestamp      power  current   voltage  frequency  power_factor
1514744820.0 7881.40234 11.54801 242.28492   50.00787       0.94180
1514744880.0 7851.73950 11.50060 242.14083   50.01876       0.94231
1514744940.0 7845.76758 11.49999 242.24704   50.02354       0.94140



girls_ups  (girls_hostel_ups.csv)   building: Girls_Hostel
columns: ['timestamp', 'power', 'current', 'voltage', 'frequency', 'power_factor']
  has frequency column: yes

first 3 rows:
 timestamp      power  current   voltage  frequency  power_factor
1376073000 6324.17847      NaN 244.01835   50.32590      -0.99594
1376073060 6348.66260      NaN 244.15110   50.35921      -0.99613
1376073120 6384.14258      NaN 244.38654   50.40094      -0.99458

last 3 rows:
   timestamp      power  current   voltage  frequency  power_factor
1514744820.0 5000.80591  7.67478 242.57769   50.00563           1.0
1514744880.0 5005.22241  7.67917 242.53189   50.02183           1.0
1514744940.0 5007.37964  7.68283 242.58704   50.01916           1.0



library_mains  (library_build_mains.csv)   building: Library
columns: ['timestamp', 'power', 'current', 'voltage', 'frequency', 'power_factor']
  has frequency column: yes

first 3 rows:
 timestamp      power  current   voltage  frequency  power_factor
1376073000 3972.19702      NaN 241.11909   50.32545      -0.99338
1376073060 3972.83789      NaN 241.30883   50.35849      -0.99350
1376073120 3974.43042      NaN 241.47595   50.40181      -0.99338

last 3 rows:
   timestamp      power  current   voltage  frequency  power_factor
1514744820.0 3303.29700  5.08487 239.42738   50.00447      -0.97775
1514744880.0 3297.89343  5.07943 239.35323   50.02238      -0.97774
1514744940.0 3299.53235  5.08129 239.39713   50.01865      -0.97775



mess_mains  (mess_build_mains.csv)   building: Mess
columns: ['timestamp', 'power', 'current', 'voltage', 'frequency', 'power_factor']
  has frequency column: yes

first 3 rows:
 timestamp       power  current   voltage  frequency  power_factor
1379961000 20197.85156      NaN 241.79240   49.92020       0.92795
1379961060 26680.30664      NaN 241.36016   49.89456       0.89957
1379961120 27281.28418      NaN 241.08492   49.84626       0.89916

last 3 rows:
 timestamp       power  current   voltage  frequency  power_factor
1514744820 10298.10352 14.63542 243.07374   50.00747       0.98087
1514744880 10358.43604 14.66883 242.86589   50.02068       0.97123
1514744940 10365.40332 14.67267 242.97758   50.02366       0.97137



lecture_mains  (lecture_build_mains.csv)   building: Lecture
columns: ['timestamp', 'power', 'current', 'voltage', 'frequency', 'power_factor']
  has frequency column: yes

first 3 rows:
 timestamp  power  current  voltage  frequency  power_factor
1376073000      0      NaN      NaN   50.32686             0
1376073060      0      NaN      NaN   50.36170             0
1376073120      0      NaN      NaN   50.40117             0

last 3 rows:
   timestamp  power  current   voltage  frequency  power_factor
1514744820.0    0.0      0.0 239.50384   50.00635           0.0
1514744880.0    0.0      0.0 239.38370   50.01825           0.0
1514744940.0    0.0      0.0 239.48580   50.02482           0.0



facilities_mains  (facilities_build_mains.csv)   building: Facilities
columns: ['timestamp', 'power', 'current', 'voltage', 'frequency', 'power_factor']
  has frequency column: yes

first 3 rows:
 timestamp      power  current   voltage  frequency  power_factor
1384453800 3390.47437      NaN 244.84043   50.28713       0.97416
1384453860 3386.13733      NaN 244.67918   50.24261       0.97416
1384453920 3383.97070      NaN 244.56806   50.21801       0.97408

last 3 rows:
   timestamp       power  current   voltage  frequency  power_factor
1514744820.0 10109.86230 14.38321 239.16866   50.00848           1.0
1514744880.0 10025.46826 14.30592 239.05692   50.02047           1.0
1514744940.0 10104.07617 14.41421 239.16097   50.02600           1.0



**What we found.** Every building meter has the same six columns: `timestamp`,
`power`, `current`, `voltage`, `frequency`, `power_factor`. The first rows show
`current` as `NA` -- the dataset readme warns about this. The timestamps are
large integers (UNIX seconds), so nothing is human-readable until we convert.

## Step 5: how much of the time range does each meter actually cover?

A file having 2.2 million rows does not mean it has 2.2 million *consecutive*
minutes. For each meter we compute:

* the first and last timestamp, converted to India time
* how many 1-minute readings there *should* be over that span
* how many there actually are (row coverage %)
* how many gaps longer than one minute there are, and the biggest one
* the data-quality counts: zero power, negative power factor, out-of-range voltage

This is the table that tells us what Phase 1 has to fix.

In [9]:
energy_profiles = pd.DataFrame(
    [
        explore.profile_energy_csv(C.meter_path(mk), C.building_of_meter(mk))
        for mk in C.METERS
    ]
)
energy_profiles.insert(0, "meter", list(C.METERS))
display(energy_profiles[[
    "meter", "label", "rows", "first", "last", "span_days",
    "row_coverage_pct", "gaps_over_1min", "largest_gap_hours",
]])

,meter,label,rows,first,last,span_days,row_coverage_pct,gaps_over_1min,largest_gap_hours
0,acad_mains,Academic,2269530,2013-08-10 00:00:00+05:30,2017-12-31 23:59:00+05:30,1605,98.20,489,254.7
1,boys_mains,Boys_Hostel,1671546,2013-08-10 00:00:00+05:30,2017-12-31 23:59:00+05:30,1605,72.32,9149,4987.8
2,boys_ups,Boys_Hostel,1688044,2013-08-10 00:00:00+05:30,2017-12-31 23:59:00+05:30,1605,73.04,9783,4987.8
3,girls_mains,Girls_Hostel,1653053,2013-08-10 00:00:00+05:30,2017-12-31 23:59:00+05:30,1605,71.52,616,5311.0
4,girls_ups,Girls_Hostel,2300745,2013-08-10 00:00:00+05:30,2017-12-31 23:59:00+05:30,1605,99.55,563,32.1
5,library_mains,Library,1713637,2013-08-10 00:00:00+05:30,2017-12-31 23:59:00+05:30,1605,74.14,639,4890.8
6,mess_mains,Mess,2013090,2013-09-24 00:00:00+05:30,2017-12-31 23:59:00+05:30,1560,89.61,1074,1465.2
7,lecture_mains,Lecture,2269309,2013-08-10 00:00:00+05:30,2017-12-31 23:59:00+05:30,1605,98.19,719,254.7
8,facilities_mains,Facilities,2048910,2013-11-15 00:00:00+05:30,2017-12-31 23:59:00+05:30,1508,94.35,3191,1054.8


In [10]:
display(energy_profiles[[
    "meter", "power_min_w", "power_mean_w", "power_max_w", "power_zero_pct",
    "pf_negative_pct", "pf_min", "pf_max", "voltage_min", "voltage_max",
    "voltage_out_of_range_pct", "current_na_pct",
]])

energy_profiles.to_csv(C.RESULTS_DIR / "phase0_energy_profile.csv", index=False)
print("saved results/phase0_energy_profile.csv")

,meter,power_min_w,power_mean_w,power_max_w,power_zero_pct,pf_negative_pct,pf_min,pf_max,voltage_min,voltage_max,voltage_out_of_range_pct,current_na_pct
0,acad_mains,0.0,27785.8,95786.3,0.29,13.78,-0.999,1.0,0.00,269.17,0.003,9.29
1,boys_mains,0.0,17488.0,97663.9,0.01,43.64,-0.999,1.0,0.00,281.76,0.005,12.56
2,boys_ups,0.0,13927.4,47212.1,0.00,47.46,-0.999,1.0,0.00,277.74,0.083,12.45
3,girls_mains,0.0,7315.9,20895.9,0.00,67.55,-0.999,1.0,0.00,280.90,0.002,11.38
4,girls_ups,0.0,6881.3,18126.8,0.01,66.05,-0.999,1.0,0.00,277.68,0.022,9.12
5,library_mains,0.0,9074.7,99208.9,0.00,58.08,-0.999,1.0,0.00,270.41,0.001,12.34
6,mess_mains,232.5,22195.3,142442.1,0.00,0.05,-0.999,1.0,209.60,270.45,0.000,7.19
7,lecture_mains,0.0,641.7,26703.1,81.70,0.01,-0.996,1.0,119.51,254.17,0.000,9.29
8,facilities_mains,0.0,10458.9,139437.1,0.00,3.14,-0.999,1.0,0.00,268.56,0.003,3.24


saved results/phase0_energy_profile.csv


**What we found -- and these findings drive the whole of Phase 1.**

1. **The Lecture building meter is mostly dead.** Over 80% of its readings are
   exactly 0 W. A lecture hall does use little power at night, but not exactly
   zero for months on end -- that is a meter that stopped reporting. We must not
   treat those zeros as "the building used no electricity".
2. **Negative `power_factor` is a sign convention, not corruption.** Between
   about 14% and 66% of readings per meter are negative, the range is exactly
   -0.999 to +1.000, and `power` itself is *never* negative. If electricity were
   genuinely flowing backwards, power would be negative too. So the sign is a
   leading/lagging indicator, and the usable value is the magnitude. Throwing
   these rows away would delete more than half of the Library data.
3. **`current` is missing for roughly 9-13% of rows**, all at the start of the
   record, exactly as the readme warns.
4. **Voltage is almost always fine** -- under 0.1% of readings fall outside a
   credible 180-270 V band, so this is a small, safe fix.
5. **Coverage is high but gaps are long.** Most meters cover 97-99% of their
   span row-wise, but individual gaps run to several months. Counting "percent
   of rows present" alone would hide that, so Phase 1 puts the data on a
   complete time grid before measuring anything.
6. **The meters do not all start together.** The Mess starts in September 2013
   and Facilities in November 2013, months after the rest.

## Step 6: the occupancy files

Occupancy is a count of devices seen by the building's WiFi access points,
recorded every 10 minutes. Two things to check: the time span (because it is
shorter than the energy data, and the overlap is our working period), and
whether the count ever reaches zero.

In [11]:
for building in C.BUILDING_ORDER:
    path = C.occupancy_path(building)
    head, tail = explore.head_tail(path, n=3)
    print("=" * 78)
    print(f"{building}  ({path.name})")
    print("=" * 78)
    print("first 3 rows:")
    print(head.to_string(index=False))
    print("last 3 rows:")
    print(tail.to_string(index=False))
    print()

Academic  (ACB.csv)
first 3 rows:
   timestamp  occupancy_count
1392490200.0             20.0
1392490800.0             28.0
1392491400.0             32.0
last 3 rows:
   timestamp  occupancy_count
1509732000.0             36.0
1509732600.0             36.0
1509733200.0             36.0

Boys_Hostel  (BH.csv)
first 3 rows:
   timestamp  occupancy_count
1392489600.0             32.0
1392490200.0             60.0
1392490800.0             88.0
last 3 rows:
   timestamp  occupancy_count
1509732000.0            421.0
1509732600.0            421.0
1509733200.0            421.0

Girls_Hostel  (GH.csv)
first 3 rows:
   timestamp  occupancy_count
1392490200.0             40.0
1392490800.0             56.0
1392491400.0             56.0
last 3 rows:
   timestamp  occupancy_count
1509732000.0            160.0
1509732600.0            160.0
1509733200.0            160.0

Mess  (DB.csv)
first 3 rows:
   timestamp  occupancy_count
1392490800.0              1.0
1392491400.0              3.0
1392492000.0

Library  (LB.csv)
first 3 rows:
   timestamp  occupancy_count
1392490200.0              4.0
1392490800.0              5.0
1392491400.0              5.0
last 3 rows:
   timestamp  occupancy_count
1509732000.0             86.0
1509732600.0             86.0
1509733200.0             86.0



Lecture  (LCB.csv)
first 3 rows:
   timestamp  occupancy_count
1392495600.0              1.0
1392496200.0              1.0
1392496800.0              1.0
last 3 rows:
   timestamp  occupancy_count
1509732000.0              1.0
1509732600.0              1.0
1509733200.0              1.0

Facilities  (SRB.csv)
first 3 rows:
   timestamp  occupancy_count
1404973800.0              4.0
1404974400.0              5.0
1404975000.0              6.0
last 3 rows:
   timestamp  occupancy_count
1509732000.0              2.0
1509732600.0              2.0
1509733200.0              2.0



In [12]:
occ_profiles = pd.DataFrame(
    [explore.profile_occupancy_csv(C.occupancy_path(b), b) for b in C.BUILDING_ORDER]
)
display(occ_profiles[[
    "building", "file", "rows", "first", "last", "span_days",
    "row_coverage_pct", "aligned_to_10min",
]])

,building,file,rows,first,last,span_days,row_coverage_pct,aligned_to_10min
0,Academic,ACB.csv,179701,2014-02-16 00:20:00+05:30,2017-11-03 23:50:00+05:30,1357,91.96,True
1,Boys_Hostel,BH.csv,179849,2014-02-16 00:10:00+05:30,2017-11-03 23:50:00+05:30,1357,92.04,True
2,Girls_Hostel,GH.csv,179681,2014-02-16 00:20:00+05:30,2017-11-03 23:50:00+05:30,1357,91.95,True
3,Mess,DB.csv,178427,2014-02-16 00:30:00+05:30,2017-11-03 23:50:00+05:30,1357,91.31,True
4,Library,LB.csv,173006,2014-02-16 00:20:00+05:30,2017-11-03 23:50:00+05:30,1357,88.54,True
5,Lecture,LCB.csv,151516,2014-02-16 01:50:00+05:30,2017-11-03 23:50:00+05:30,1357,77.54,True
6,Facilities,SRB.csv,157649,2014-07-10 12:00:00+05:30,2017-11-03 23:50:00+05:30,1213,90.29,True


In [13]:
display(occ_profiles[[
    "building", "occ_min", "occ_p05", "occ_median", "occ_mean", "occ_p95",
    "occ_max", "low_occ_threshold", "pct_rows_at_or_below_threshold",
]])

occ_profiles.to_csv(C.RESULTS_DIR / "phase0_occupancy_profile.csv", index=False)
print("saved results/phase0_occupancy_profile.csv")

,building,occ_min,occ_p05,occ_median,occ_mean,occ_p95,occ_max,low_occ_threshold,pct_rows_at_or_below_threshold
0,Academic,1,11.0,44.0,78.4,257.0,505,12.85,6.51
1,Boys_Hostel,1,29.0,221.0,222.1,421.0,614,21.05,4.13
2,Girls_Hostel,1,11.0,97.0,95.7,178.0,275,8.90,4.02
3,Mess,1,4.0,44.0,58.5,174.0,340,8.70,12.23
4,Library,1,2.0,21.0,41.7,160.0,498,8.00,27.30
5,Lecture,1,1.0,6.0,39.9,209.0,498,10.45,60.39
6,Facilities,1,1.0,5.0,6.6,18.0,47,0.90,0.00


saved results/phase0_occupancy_profile.csv


**What we found -- this is the single most important finding in Phase 0.**

**Occupancy never reaches zero.** The minimum count in every one of the seven
buildings is 1, not 0. This is the WiFi over-counting the dataset authors warn
about: idle phones and laptops stay associated to an access point even when
nobody is using them. So the main research question cannot be phrased as "energy
used when the building is empty" -- there is no such reading. It has to be
"energy used when occupancy is **low**", with a threshold we state openly.

We use a *relative* threshold: low occupancy means at or below 5% of that
building's own 95th-percentile occupancy. The `low_occ_threshold` column shows
what that works out to, and `pct_rows_at_or_below_threshold` shows how much data
it captures. Two buildings need special handling:

* **Facilities (SRB)** is a small building whose occupancy runs 1 to 47, so 5% of
  its 95th percentile is 0.9 -- below its own minimum of 1. **No reading
  qualifies.** We keep the same rule for every building rather than bending the
  definition, report this cell honestly, and read Facilities off the threshold
  sensitivity curve in Phase 5 instead.
* **Lecture (LCB)** has 60% of its readings under the threshold, which combined
  with its dead meter means its waste figure must be computed over the periods
  when the meter was alive.

Reassuringly, every occupancy timestamp sits exactly on a 10-minute boundary, so
energy and occupancy will line up without any fuzzy matching.

## Step 7: the two combined files

`all_buildings_power.csv` holds the power of all nine meters side by side, and
`data_present_status_buildings.csv` holds a 1/0 flag per minute per building.
We are not using them as our main source -- they have no voltage or power factor
-- but they are valuable:

* the power file gives us a completely **independent way to check** the totals we
  compute from the individual meter files
* the status file is the cleanest way to build a **missing-data heatmap**

In [14]:
for path in (C.ALL_BUILDINGS_POWER_CSV, C.STATUS_BUILDINGS_CSV):
    head = pd.read_csv(path, nrows=3)
    print("=" * 78)
    print(path.name)
    print("=" * 78)
    print("columns:", list(head.columns))
    print(head.to_string(index=False))
    print()

all_buildings_power.csv
columns: ['timestamp', 'Academic', 'Boys_main', 'Boys_backup', 'Facilities', 'Girls_main', 'Girls_backup', 'Lecture', 'Library', 'Mess']
 timestamp    Academic   Boys_main  Boys_backup  Facilities  Girls_main  Girls_backup  Lecture    Library  Mess
1376073000 28368.60645 28298.05859  11521.14258         NaN  9817.66846    6324.17847        0 3972.19702   NaN
1376073060 28411.98828 28327.11426  11532.83105         NaN  9804.43408    6348.66260        0 3972.83789   NaN
1376073120 28752.53809 28343.01562  11538.75635         NaN  9834.51758    6384.14258        0 3974.43042   NaN

data_present_status_buildings.csv
columns: ['timestamp', 'Academic', 'Boys_main', 'Boys_backup', 'Facilities', 'Girls_main', 'Girls_backup', 'Lecture', 'Library', 'Mess']
          timestamp  Academic  Boys_main  Boys_backup  Facilities  Girls_main  Girls_backup  Lecture  Library  Mess
2013-08-10 00:00:00         1          1            1           0           1             1        1   

Note the column names in these two files (`Boys_main`, `Boys_backup`, ...)
differ from our meter keys. `src/config.py` holds the translation in
`WIDE_COL_TO_METER` so the cross-check in Phase 1 is unambiguous.

## Step 8: the working period

Energy recording starts in August 2013, but occupancy only starts in February
2014 and stops in November 2017. Since every research question needs both, our
working period is the **overlap**.

In [15]:
start, end = explore.overlap_window(energy_profiles, occ_profiles)
print("energy spans   :", energy_profiles["first"].min(), "->", energy_profiles["last"].max())
print("occupancy spans:", occ_profiles["first"].min(), "->", occ_profiles["last"].max())
print()
print("WORKING PERIOD :", start, "->", end)
print(f"                 {(end - start).days} days, "
      f"{int((end - start).total_seconds() // 600):,} ten-minute steps")

energy spans   : 2013-08-10 00:00:00+05:30 -> 2017-12-31 23:59:00+05:30
occupancy spans: 2014-02-16 00:10:00+05:30 -> 2017-11-03 23:50:00+05:30

WORKING PERIOD : 2014-02-16 00:10:00+05:30 -> 2017-11-03 23:50:00+05:30
                 1356 days, 195,406 ten-minute steps


## Step 9: a picture of when each meter was actually recording

The table above is precise but hard to take in. The obvious chart -- one bar per
source from its first timestamp to its last -- would be **misleading**, because
a meter that went silent for 200 days in the middle still produces a solid bar.

So instead we count how many readings each source produced in each calendar
month and divide by how many that month should contain (1440 per day for the
1-minute energy meters, 144 per day for the 10-minute occupancy files). Dark
means a full month of data; pale means a month with holes; white means the
source recorded nothing at all that month.

In [16]:
import matplotlib.pyplot as plt

coverage = explore.coverage_matrix()
print(f"coverage matrix: {coverage.shape[0]} sources x {coverage.shape[1]} months")
display(coverage.iloc[:, :6].round(1))

C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()


C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()


C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()
C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()


C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()
C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()


C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()
C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()


...

C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()


C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()
C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()
C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()
C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()
C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Inde

coverage matrix: 16 sources x 53 months


C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()
C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()
C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()
C:\Users\User\Desktop\VIT\TY\Sem 5\DS\CP\src\explore.py:154: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months = to_ist(chunk["timestamp"]).dt.to_period("M").value_counts()


,2013-08,2013-09,2013-10,2013-11,2013-12,2014-01
acad_mains (energy),71.0,98.5,99.6,100.0,99.7,98.6
boys_mains (energy),70.7,97.7,99.0,99.8,99.5,99.7
boys_ups (energy),70.7,97.8,99.2,99.9,99.6,99.4
girls_mains (energy),68.6,52.0,97.5,99.8,99.0,99.2
girls_ups (energy),70.9,99.9,99.0,100.0,99.2,95.5
library_mains (energy),71.0,100.0,99.6,100.0,99.7,100.0
mess_mains (energy),NaN,23.2,99.0,96.4,99.9,96.2
lecture_mains (energy),70.9,98.4,99.6,100.0,99.7,98.5
facilities_mains (energy),NaN,NaN,NaN,43.6,96.6,100.0
Academic (occupancy),NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
fig, ax = plt.subplots(figsize=(13, 6))

# A month with no rows at all is a different thing from a month that recorded
# almost nothing, so NaN gets its own neutral grey rather than the palest blue.
cmap = viz.SEQ_BLUE.copy()
cmap.set_bad("#e8e7e3")

im = ax.imshow(
    coverage.values, aspect="auto", cmap=cmap, vmin=0, vmax=100,
    interpolation="nearest",
)

ax.set_yticks(range(len(coverage.index)))
ax.set_yticklabels(coverage.index, fontsize=8)

# Label every third month so the axis stays readable.
tick_positions = range(0, len(coverage.columns), 3)
ax.set_xticks(list(tick_positions))
ax.set_xticklabels([coverage.columns[i] for i in tick_positions], rotation=45,
                   ha="right", fontsize=8)

# Separate the energy block from the occupancy block.
ax.axhline(len(C.METERS) - 0.5, color=viz.INK, linewidth=1.2)

# Mark where the usable overlap begins and ends.
months = list(coverage.columns)
for boundary, text in (
    (str(start.to_period("M")), "occupancy starts"),
    (str(end.to_period("M")), "occupancy ends"),
):
    if boundary in months:
        x = months.index(boundary)
        ax.axvline(x, color=viz.STATUS["ANOMALY"], linewidth=1.4, linestyle="--")
        ax.annotate(text, xy=(x, -0.8), fontsize=8, color=viz.STATUS["ANOMALY"],
                    ha="center", va="bottom", annotation_clip=False)

# pad lifts the title clear of the two dashed-line annotations above the plot
ax.set_title("How complete each data source is, month by month", pad=26)
ax.grid(False)
cbar = fig.colorbar(im, ax=ax, pad=0.01, fraction=0.025)
cbar.set_label("% of expected readings present", fontsize=9,
               color=viz.INK_SECONDARY)
cbar.outline.set_visible(False)

# Grey means the source recorded nothing at all that month -- distinct from the
# palest blue, which means it recorded something but almost nothing.
from matplotlib.patches import Patch
ax.legend(
    handles=[Patch(facecolor="#e8e7e3", label="no data recorded at all")],
    loc="upper left", bbox_to_anchor=(0.0, -0.30), fontsize=8,
)

viz.save_fig(fig, "fig_00_coverage_timeline")
coverage.round(1).to_csv(C.RESULTS_DIR / "phase0_monthly_coverage.csv")
print("saved results/phase0_monthly_coverage.csv")

C:\Users\User\AppData\Local\Temp\ipykernel_20756\2484399202.py:28: UserWarning: Converting to Period representation will drop timezone information.
  (str(start.to_period("M")), "occupancy starts"),
C:\Users\User\AppData\Local\Temp\ipykernel_20756\2484399202.py:29: UserWarning: Converting to Period representation will drop timezone information.
  (str(end.to_period("M")), "occupancy ends"),


saved  figures/fig_00_coverage_timeline.png   ->  ![...](../figures/fig_00_coverage_timeline.png)
saved results/phase0_monthly_coverage.csv


**Takeaway:** the record is far patchier than the row counts suggest. The Boys
hostel, Girls mains and Library meters each have a multi-month white band where
nothing was recorded at all, which is why their row coverage sits near 72-74%
rather than 98%. The occupancy block (below the black line) starts only in
February 2014 and stops in November 2017 -- the two dashed lines. Everything
this project analyses lives between them.

In [18]:
# Which months are completely missing for each source? Worth naming explicitly,
# because these are the periods that will simply be absent from every result.
gaps = []
for source in coverage.index:
    row = coverage.loc[source]
    blank = row[row.isna() | (row < 1)].index.tolist()
    if blank:
        gaps.append({"source": source, "blank months": len(blank),
                     "from": blank[0], "to": blank[-1]})
display(pd.DataFrame(gaps))

,source,blank months,from,to
0,boys_mains (energy),9,2015-06,2016-05
1,boys_ups (energy),9,2015-06,2016-05
2,girls_mains (energy),9,2015-06,2016-06
3,library_mains (energy),8,2014-08,2015-07
4,mess_mains (energy),3,2013-08,2016-05
5,facilities_mains (energy),4,2013-08,2015-10
6,Academic (occupancy),8,2013-08,2017-12
7,Boys_Hostel (occupancy),8,2013-08,2017-12
8,Girls_Hostel (occupancy),8,2013-08,2017-12
9,Mess (occupancy),8,2013-08,2017-12


## Step 10: the Phase 0 data-quality summary

Finally we pull the findings into one small table. This is what goes into
section 4 of `docs/PROJECT_REPORT.md`, and it is the to-do list for Phase 1.

In [19]:
issues = pd.DataFrame(
    [
        {
            "issue": "Negative power_factor",
            "where": "all 9 meters, 14-66% of rows",
            "evidence": "range exactly -0.999..+1.000 while power is never negative",
            "reading": "sign convention (leading/lagging), not bad data",
            "phase 1 action": "keep abs(power_factor); keep a pf_was_negative flag",
        },
        {
            "issue": "Dead meter (0 W for months)",
            "where": "Lecture building, 81.7% of rows",
            "evidence": "exact zeros in long unbroken runs",
            "reading": "meter stopped reporting, not zero consumption",
            "phase 1 action": "flag runs of 0 W longer than 6 h as meter_off",
        },
        {
            "issue": "Missing current",
            "where": "all meters, 9-13% of rows, early period",
            "evidence": "NA in the first rows of every file; readme confirms",
            "reading": "instrument not configured yet",
            "phase 1 action": "leave as NaN; we do not model current",
        },
        {
            "issue": "Out-of-range voltage",
            "where": "all meters, under 0.1% of rows",
            "evidence": "values at 0 V and above 270 V on a 230 V feeder",
            "reading": "momentary meter artefact",
            "phase 1 action": "set outside 180-270 V to NaN",
        },
        {
            "issue": "Long gaps",
            "where": "Library and both hostels, up to ~208 days",
            "evidence": "largest_gap_hours column above",
            "reading": "extended outage in data collection",
            "phase 1 action": "put data on a complete time grid; interpolate only gaps <= 30 min",
        },
        {
            "issue": "Staggered start dates",
            "where": "Mess (Sep 2013), Facilities (Nov 2013)",
            "evidence": "first column above",
            "reading": "meters installed later",
            "phase 1 action": "per-building coverage reported; no back-filling",
        },
        {
            "issue": "Occupancy never reaches zero",
            "where": "all 7 buildings, minimum count = 1",
            "evidence": "occ_min column above",
            "reading": "WiFi counts idle devices, not people",
            "phase 1 / 5 action": "relative low-occupancy threshold + sensitivity curve",
        },
        {
            "issue": "Threshold unreachable for Facilities",
            "where": "Facilities (SRB)",
            "evidence": "5% of p95 = 0.9 < minimum observed count of 1",
            "reading": "small building, occupancy range 1-47",
            "phase 1 / 5 action": "keep the rule; report the cell honestly; use the sensitivity curve",
        },
    ]
)
display(issues)
issues.to_csv(C.RESULTS_DIR / "phase0_data_quality_issues.csv", index=False)
print("saved results/phase0_data_quality_issues.csv")

,issue,where,evidence,reading,phase 1 action,phase 1 / 5 action
0,Negative power_factor,"all 9 meters, 14-66% of rows",range exactly -0.999..+1.000 while power is ne...,"sign convention (leading/lagging), not bad data",keep abs(power_factor); keep a pf_was_negative...,NaN
1,Dead meter (0 W for months),"Lecture building, 81.7% of rows",exact zeros in long unbroken runs,"meter stopped reporting, not zero consumption",flag runs of 0 W longer than 6 h as meter_off,NaN
2,Missing current,"all meters, 9-13% of rows, early period",NA in the first rows of every file; readme con...,instrument not configured yet,leave as NaN; we do not model current,NaN
3,Out-of-range voltage,"all meters, under 0.1% of rows",values at 0 V and above 270 V on a 230 V feeder,momentary meter artefact,set outside 180-270 V to NaN,NaN
4,Long gaps,"Library and both hostels, up to ~208 days",largest_gap_hours column above,extended outage in data collection,put data on a complete time grid; interpolate ...,NaN
5,Staggered start dates,"Mess (Sep 2013), Facilities (Nov 2013)",first column above,meters installed later,per-building coverage reported; no back-filling,NaN
6,Occupancy never reaches zero,"all 7 buildings, minimum count = 1",occ_min column above,"WiFi counts idle devices, not people",NaN,relative low-occupancy threshold + sensitivity...
7,Threshold unreachable for Facilities,Facilities (SRB),5% of p95 = 0.9 < minimum observed count of 1,"small building, occupancy range 1-47",NaN,keep the rule; report the cell honestly; use t...


saved results/phase0_data_quality_issues.csv


## Step 11: write these findings into the project report

The project rule is that **every number in `docs/PROJECT_REPORT.md` comes from
executed code** -- nothing is typed in by hand. The report contains named
placeholder blocks, and `src/report.py` fills them from the tables we just
built. Re-running this notebook rewrites the blocks, so the report can never
drift out of step with the analysis.

In [20]:
from src import report

blocks = {}

blocks["phase0_mapping"] = report.md_table(mapping)

inventory = pd.concat(
    [
        explore.file_inventory(C.ENERGY_DIR).assign(folder="energy_dataset"),
        explore.file_inventory(C.OCCUPANCY_DIR).assign(folder="IIITD_occupancy_dataset"),
    ]
)[["folder", "file", "size_mb"]].reset_index(drop=True)
blocks["phase0_file_inventory"] = (
    report.md_table(inventory)
    + f"\n\nTotal: **{inventory['size_mb'].sum():,.0f} MB** across "
    f"{len(inventory)} files."
)

blocks["phase0_energy_profile"] = report.md_table(
    energy_profiles[[
        "meter", "label", "rows", "first", "last", "span_days",
        "row_coverage_pct", "gaps_over_1min", "largest_gap_hours",
    ]]
)

blocks["phase0_energy_quality"] = report.md_table(
    energy_profiles[[
        "meter", "power_mean_w", "power_max_w", "power_zero_pct",
        "pf_negative_pct", "pf_min", "pf_max", "voltage_min", "voltage_max",
        "voltage_out_of_range_pct", "current_na_pct",
    ]]
)

blocks["phase0_occupancy_profile"] = report.md_table(
    occ_profiles[[
        "building", "file", "rows", "first", "last", "span_days",
        "row_coverage_pct", "aligned_to_10min",
    ]]
)

blocks["phase0_occupancy_distribution"] = report.md_table(
    occ_profiles[[
        "building", "occ_min", "occ_p05", "occ_median", "occ_mean", "occ_p95",
        "occ_max", "low_occ_threshold", "pct_rows_at_or_below_threshold",
    ]]
)

print("\n".join(f"prepared block: {k}" for k in blocks))

prepared block: phase0_mapping
prepared block: phase0_file_inventory
prepared block: phase0_energy_profile
prepared block: phase0_energy_quality
prepared block: phase0_occupancy_profile
prepared block: phase0_occupancy_distribution


In [21]:
# The written findings that go with those tables. These are conclusions, not
# numbers, so they are prose -- but every figure quoted inside them is read from
# the tables above rather than typed.
srb = occ_profiles.set_index("building").loc["Facilities"]
lcb = occ_profiles.set_index("building").loc["Lecture"]
lecture_zero = float(
    energy_profiles.set_index("meter").loc["lecture_mains", "power_zero_pct"]
)
pf_lo = energy_profiles["pf_negative_pct"].min()
pf_hi = energy_profiles["pf_negative_pct"].max()

blocks["phase0_occupancy_note"] = f"""
**The single most important finding in exploration: occupancy never reaches zero.**
The minimum count in every one of the seven buildings is **1**, not 0. This is the
WiFi over-counting the dataset authors warn about -- idle phones and laptops stay
associated with an access point long after their owner has left. The consequence is
structural, not cosmetic: *the main research question cannot be phrased as "energy
used while the building is empty", because no such reading exists in this dataset.*

It must instead be "energy used while occupancy is **low**", against a threshold we
state openly. We use a threshold relative to each building's own scale:
**low occupancy = occupancy at or below {C.LOW_OCC_FRACTION:.0%} of that building's
{C.LOW_OCC_PERCENTILE}th-percentile occupancy.** An absolute cut-off would be
meaningless across buildings whose normal populations differ by a factor of twenty.

Two buildings do not fit the standard recipe, and both are reported rather than
quietly dropped:

- **Facilities (SRB)** is small: its occupancy runs {srb['occ_min']:.0f} to
  {srb['occ_max']:.0f} with a 95th percentile of {srb['occ_p95']:.0f}, so the
  threshold works out to **{srb['low_occ_threshold']:.2f}** -- below its own
  minimum observed count of {srb['occ_min']:.0f}. **No reading qualifies**
  ({srb['pct_rows_at_or_below_threshold']:.2f}% of rows). We keep the identical
  rule for every building rather than bending the definition for one, report this
  cell honestly as "no qualifying intervals", and read Facilities off the
  threshold-sensitivity curve in Phase 5 instead.
- **Lecture (LCB)** has {lcb['pct_rows_at_or_below_threshold']:.1f}% of its
  readings under the threshold, and its meter reads exactly 0 W for
  {lecture_zero:.1f}% of the record. Its waste figure is therefore computed only
  over the periods when the meter was demonstrably alive, with the coverage
  reported alongside.

Reassuringly, **every occupancy timestamp sits exactly on a 10-minute boundary**,
so energy and occupancy line up without any fuzzy time matching.
"""

blocks["phase0_coverage_figure"] = report.figure(
    "fig_00_coverage_timeline",
    "Monthly completeness of every energy meter and occupancy file",
    "The record is far patchier than the row counts suggest: the Boys hostel, "
    "Girls mains and Library meters each lose several consecutive months "
    "entirely, and the occupancy block (below the black line) exists only "
    "between the two dashed lines.",
)

blocks["phase0_data_quality_issues"] = report.md_table(issues)

blocks["phase0_working_period"] = f"""
Energy recording runs from **{energy_profiles['first'].min():%Y-%m-%d}** to
**{energy_profiles['last'].max():%Y-%m-%d}**, but occupancy only exists from
**{occ_profiles['first'].min():%Y-%m-%d}** to **{occ_profiles['last'].max():%Y-%m-%d}**.
Every research question in this project needs both, so the working period is the
overlap:

> **{start:%d %B %Y} to {end:%d %B %Y}** -- {(end - start).days:,} days,
> {int((end - start).total_seconds() // 600):,} ten-minute intervals.

The roughly six months of energy data that precede the occupancy record are not
used. They are not deleted, simply out of scope for questions that require knowing
whether anyone was in the building.
"""

print("\n".join(f"prepared block: {k}" for k in blocks))

prepared block: phase0_mapping
prepared block: phase0_file_inventory
prepared block: phase0_energy_profile
prepared block: phase0_energy_quality
prepared block: phase0_occupancy_profile
prepared block: phase0_occupancy_distribution
prepared block: phase0_occupancy_note
prepared block: phase0_coverage_figure
prepared block: phase0_data_quality_issues
prepared block: phase0_working_period


In [22]:
blocks["method_phase0"] = f"""
Exploration was done before any cleaning, to find out what the data actually
contains rather than what the documentation promises.

**What we did.** We listed every file with its size; read both `Readme.txt` files
and the ISA-Tab metadata that records the instruments used; printed the first and
last rows of all {len(C.METERS)} energy meters and all {len(C.BUILDINGS)} occupancy
files; and profiled each file for coverage, gaps and data-quality counts.

**How we handled the size.** The energy folder is about 1.5 GB, so no step ever
loads it all. Files are read one at a time, in chunks of 500,000 rows, keeping
only the needed columns. Reading the last rows of a 125 MB file, for example, is
done by streaming through it and keeping only the final chunk.

**Timestamps.** All UNIX timestamps are converted with
`pd.to_datetime(..., unit="s", utc=True).dt.tz_convert("{C.TIMEZONE}")`, which is
what the dataset readme requires. Getting this wrong by 5.5 hours would put every
"night-time" reading in the afternoon and silently invalidate the entire project.

**Coverage measurement.** Rather than drawing a bar from each meter's first to its
last timestamp -- which makes a meter that went silent for 200 days look
continuous -- we counted readings per calendar month and divided by how many that
month should contain (1440 per day for the 1-minute energy files, 144 per day for
the 10-minute occupancy files). That is what the heatmap in section 4.8 shows.

**Notebook:** `notebooks/00_explore.ipynb`.
"""

blocks["results_phase0"] = f"""
Exploration produced four findings that shaped everything after it.

1. **The working period is set by occupancy, not energy.** Energy covers
   {energy_profiles['first'].min():%b %Y} – {energy_profiles['last'].max():%b %Y},
   occupancy only {occ_profiles['first'].min():%b %Y} – {occ_profiles['last'].max():%b %Y}.
   The project analyses the overlap.

2. **"Empty" is not measurable in this dataset.** Minimum occupancy is 1 in all
   seven buildings (section 4.7). The main question had to be re-specified around a
   relative low-occupancy threshold, with a sensitivity curve to show how much the
   answer depends on where that threshold is put.

3. **Negative `power_factor` is a sign convention, not corrupt data.** It affects
   {pf_lo:.2f}%–{pf_hi:.2f}% of readings depending on the meter. The decisive
   evidence is that the range is exactly −0.999 to +1.000 *while `power` itself is
   never negative*: if current were genuinely flowing backwards, power would be
   negative too. Treating these as invalid would have deleted more than half of the
   Library record. We keep the magnitude and retain the sign as a separate flag.

4. **Coverage is much worse than row counts suggest.** Row coverage ranges from
   {energy_profiles['row_coverage_pct'].min():.1f}% to
   {energy_profiles['row_coverage_pct'].max():.1f}%, and the worst meters lose
   whole months at a time -- the largest single gap is
   {energy_profiles['largest_gap_hours'].max():,.0f} hours
   ({energy_profiles['largest_gap_hours'].max() / 24:,.0f} days). The Lecture meter
   additionally reads exactly 0 W for {lecture_zero:.1f}% of its record, which is a
   dead meter rather than an idle building.
"""

report.update_blocks(blocks)

report: updated block 'phase0_mapping' (9 lines)
report: updated block 'phase0_file_inventory' (31 lines)
report: updated block 'phase0_energy_profile' (11 lines)


report: updated block 'phase0_energy_quality' (11 lines)


report: updated block 'phase0_occupancy_profile' (9 lines)


report: updated block 'phase0_occupancy_distribution' (9 lines)


report: updated block 'phase0_occupancy_note' (33 lines)
report: updated block 'phase0_coverage_figure' (3 lines)
report: updated block 'phase0_data_quality_issues' (10 lines)


report: updated block 'phase0_working_period' (13 lines)


report: updated block 'method_phase0' (26 lines)


report: updated block 'results_phase0' (28 lines)


## Step 12: record the Phase 0 decisions

Any time we make a judgement call, it goes into the decision log with the
options we considered and what difference the choice makes. This is what lets a
reader disagree with us precisely rather than vaguely.

In [23]:
report.log_decision(
    id="D00-01", phase="0",
    decision="Analysis window",
    options_considered="Use the full energy record (Aug 2013 - Dec 2017); "
                       "use only the energy/occupancy overlap",
    chosen=f"Overlap only: {start:%Y-%m-%d} to {end:%Y-%m-%d}",
    reason="All three research questions need occupancy. Energy-only months "
           "cannot answer any of them.",
    effect_on_results="Drops ~6 months of energy data at the start and ~2 months "
                      "at the end. Reduces sample size; does not bias it.",
)

report.log_decision(
    id="D00-02", phase="0",
    decision="Interpretation of negative power_factor",
    options_considered="Treat as invalid and drop the rows; set to NaN; "
                       "take the absolute value and keep a sign flag",
    chosen="abs(power_factor), plus a retained pf_was_negative flag",
    reason=f"Affects {pf_lo:.2f}-{pf_hi:.2f}% of rows. The range is exactly "
           "-0.999..+1.000 while power is never negative, so the sign is a "
           "leading/lagging convention, not reverse power flow.",
    effect_on_results="Preserves 58% of the Library record that dropping would "
                      "have destroyed. Power itself is untouched, so energy "
                      "totals are unaffected either way.",
)

report.log_decision(
    id="D00-03", phase="0",
    decision="Definition of low occupancy",
    options_considered="occupancy == 0 (impossible: minimum is 1); a fixed "
                       "absolute count for all buildings; a per-building "
                       "fraction of its own 95th percentile",
    chosen=f"occupancy <= {C.LOW_OCC_FRACTION:.0%} of the building's "
           f"{C.LOW_OCC_PERCENTILE}th percentile",
    reason="WiFi never reads zero, and building populations differ by ~20x, so "
           "an absolute cut-off is not comparable across buildings.",
    effect_on_results="This threshold IS the main result's definition. A "
                      "sensitivity curve over 0-20% is published in Phase 5 so "
                      "the reader can see how much the headline depends on it.",
)

report.log_decision(
    id="D00-04", phase="0",
    decision="Facilities (SRB), where no reading meets the threshold",
    options_considered="Add a percentile floor for all buildings; use an "
                       "absolute rule for SRB only; keep the uniform rule and "
                       "report the empty cell",
    chosen="Keep the uniform rule; report 'no qualifying intervals'; quote SRB "
           "from the sensitivity curve instead",
    reason=f"5% of its 95th percentile is {srb['low_occ_threshold']:.2f}, below "
           f"its own minimum count of {srb['occ_min']:.0f}. Changing the "
           "definition for one building would make it non-comparable.",
    effect_on_results="Facilities has no headline waste percentage at the "
                      "standard threshold. Its behaviour is reported from the "
                      "sensitivity curve, clearly labelled as such.",
)

report.log_decision(
    id="D00-05", phase="0",
    decision="Lecture building, whose meter reads 0 W for most of the record",
    options_considered="Exclude the building; treat the zeros as genuine zero "
                       "consumption; compute over alive periods only",
    chosen="Include it, computing over periods where the meter was alive, with "
           "coverage reported alongside",
    reason=f"{lecture_zero:.1f}% of readings are exactly 0 W in long unbroken "
           "runs -- a meter that stopped reporting, not a building using no "
           "electricity. Treating them as real would produce a near-zero waste "
           "figure that is an artefact.",
    effect_on_results="Lecture keeps a place in the headline table, but its "
                      "figure rests on a much smaller sample than the other six "
                      "buildings, and is flagged as such.",
)

report.log_decision(
    id="D00-06", phase="0",
    decision="WiFi over-count of idle devices",
    options_considered="Ignore it; subtract the documented idle baseline before "
                       "thresholding and use that as the headline; use raw "
                       "counts for the headline and report a corrected variant",
    chosen="Raw counts for the headline; corrected-occupancy run reported as a "
           "robustness check in Phase 5",
    reason="The over-count (~20 devices, ~50 in Academic) is an approximate "
           "constant from the dataset paper, not a measurement. Building the "
           "headline on it would rest the main result on an estimate.",
    effect_on_results="Headline is conservative (raw counts make buildings look "
                      "more occupied than they are, so waste is understated). "
                      "The corrected run quantifies by how much.",
)

report.log_decision(
    id="D00-07", phase="0",
    decision="Analysis resolution",
    options_considered="Keep 1-minute energy and forward-fill occupancy; "
                       "average energy into 10-minute blocks to match occupancy",
    chosen=f"{C.TARGET_FREQ} blocks",
    reason="Occupancy is natively 10-minute. Up-sampling it to 1 minute would "
           "invent ten times more data than was measured. Every occupancy "
           "timestamp already falls exactly on a 10-minute boundary.",
    effect_on_results="Reduces ~2.3M rows per meter to ~231k, which is what "
                      "makes the whole project run on a laptop. No loss of "
                      "information relative to the occupancy signal.",
)

report.log_decision(
    id="D00-08", phase="0",
    decision="Random seed",
    options_considered="Unseeded; a fixed seed",
    chosen=f"seed = {C.SEED} everywhere (sampling, k-means, anomaly injection, "
           "random forest)",
    reason="Results must be reproducible by a teammate or an examiner running "
           "the notebooks again.",
    effect_on_results="None on the substance; makes every reported number exactly "
                      "reproducible.",
)

report.publish_decision_log()
print()
print("blocks still pending:", len(report.pending_blocks()))
print("figures referenced but missing:", report.check_figures())

report: updated block 'decision_log' (50 lines)

blocks still pending: 0


figures referenced but missing: []


## Phase 0 conclusion

We have opened every file, and we now know three things that change how the rest
of the project must be built:

1. **Our working period is 2014-02-16 to 2017-11-03**, set by the occupancy data,
   not by the much longer energy record.
2. **"Empty building" is not measurable** with this data, because WiFi occupancy
   never reads zero. The main research question is therefore about *low*
   occupancy, with a stated, relative threshold -- and we will publish a
   sensitivity curve so the reader can see how much the answer depends on it.
3. **Two buildings need care, not exclusion:** Lecture (dead meter for most of
   the record) and Facilities (too small for the standard threshold).

**Next:** Phase 1 turns these raw files into one clean 10-minute table per
building, with the fixes listed in the table above, and produces the
missing-data heatmap.